In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load in the data
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
x = mushroom.data.features 
y = mushroom.data.targets.values.ravel()
  
# drop rows with missing values
mask = x.notna().all(axis=1)
x = x[mask].reset_index(drop=True)
y = y[mask]

print(f"Dataset size after dropping mmissing values: {len(x)}")
print(f"Class distribution: {pd.Series(y).value_counts().to_dict()}")

x_tr, x_te, y_tr, y_te = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

# Part 1: Train a Custom naive Bayes Classifier
class NaiveBayesClassifier:
    """Categorical Naive Bayes with Laplace smoothing"""

    def __init__(self, alpha=1.0):
        self.alpha = alpha      # Laplace smoothing parameter
        self.classes_ = None
        self.prior_ = {}         # P(Y = c) 
        self.cond_prob_ = {}    # P(X_i = x | Y = c)
        self.feature_names_ = None
        self.feature_categories_ = {} # unique values per feature

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.feature_names_ = X.columns.tolist()
        n_total = len(y)

        # Store the unique categories for each feature (from training data)
        for col in self.feature_names_:
            self.feature_categories_[col] = X[col].unique()

        # Prior probabilities with Laplace smoothing
        for c in self.classes_:
            numerator = np.sum(y == c) + self.alpha
            denominator = n_total + self.alpha * len(self.classes_)
            self.prior_[c] = numerator / denominator

        # Conditional Probabilties P(X_i = x | Y = c)
        for col in self.feature_names_:
            self.cond_prob_[col] = {}
            n_categories = len(self.feature_categories_[col])    
            for c in self.classes_:
                self.cond_prob_[col][c] = {}
                subset = X[col][y == c]
                n_class = len(subset)
                value_counts = subset.value_counts()
                for val in self.feature_categories_[col]:
                    count = value_counts.get(val, 0)
                    numerator = count + self.alpha
                    denominator = n_class + self.alpha * n_categories
                    self.cond_prob_[col][c][val] = numerator / denominator

        return self
    
    def predict(self, X):
        predictions = []
        for _, row in X.iterrows():
            posteriors = {}
            for c in self.classes_:
                # Work in log space to avoid underflow
                log_post = np.log(self.prior_[c])
                for col in self.feature_names_:
                    val = row[col]
                    if val in self.cond_prob_[col][c]:
                        log_post += np.log(self.cond_prob_[col][c][val])
                    else:
                        # Unseen category: use Laplace smoothing
                        n_categories = len(self.feature_categories_[col])
                        n_class = sum(1 for v in self.cond_prob_[col][c].values())
                        log_post += np.log(self.alpha / (n_class + self.alpha * n_categories))
                posteriors[c] = log_post
            predictions.append(max(posteriors, key=posteriors.get))
        return np.array(predictions)

# Train the custom classifier
nb_custom = NaiveBayesClassifier(alpha=1.0)
nb_custom.fit(x_tr, y_tr)

# print out the prior probabilities
print("="*60)
print("Part 1: Prior probabilities (Custom NB)")
print("="*60)
for c in nb_custom.classes_:
    label = "Edible" if c == 'e' else "Poisonous"
    print(f"\n  Class = {label}:")
    for val, prob in sorted(nb_custom.cond_prob_['odor'][c].items()):
        print(f"    P(Odor={val} | {label}) = {prob:.6f}")

# Part 2 - Predict on the test set
print("\n" + "=" * 60)
print("Part 2: Predictions on test set (Custom NB)")
print("=" * 60)

y_pred_custom = nb_custom.predict(x_te)

print(f"    Total test samples: {len(y_te)}")
print(f"    Predicted Edible: {np.sum(y_pred_custom == 'e')}")
print(f"    Predicted Poisonous: {np.sum(y_pred_custom == 'p')}")

# Part 3: evaluation metrics (custom NB)
print("\n" + "=" * 60)
print("Part 3: Evaluation Metrics (Custom NB)")
print("=" * 60)

# Treat 'p' as the positive class
acc_custom = accuracy_score(y_te, y_pred_custom)
prec_custom = precision_score(y_te, y_pred_custom, pos_label='p')
rec_custom = recall_score(y_te, y_pred_custom, pos_label='p')
f1_custom = f1_score(y_te, y_pred_custom, pos_label='p')

print(f"    Accuracy:  {acc_custom:.6f}")
print(f"    Precision: {prec_custom:.6f}")
print(f"    Recall:    {rec_custom:.6f}")
print(f"    F1 Score:  {f1_custom:.6f}")

# Part 4: compare with sklearn CategoricalNB
print("\n" + "=" * 60)
print("PART 4: Comparison with sklearn CategoricalNB")
print("=" * 60)

# Encode categorical features as integers for sklearn
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc = encoder.fit_transform(x_tr)
X_test_enc = encoder.transform(x_te)

# Train sklearn's CategoricalNB (alpha=1.0 = Laplace smoothing)
nb_sklearn = CategoricalNB(alpha=1.0)
nb_sklearn.fit(X_train_enc, y_tr)

y_pred_sklearn = nb_sklearn.predict(X_test_enc)

acc_sklearn = accuracy_score(y_te, y_pred_sklearn)
prec_sklearn = precision_score(y_te, y_pred_sklearn, pos_label='p')
rec_sklearn = recall_score(y_te, y_pred_sklearn, pos_label='p')
f1_sklearn = f1_score(y_te, y_pred_sklearn, pos_label='p')

print(f"\n  {'Metric':<12} {'Custom NB':>12} {'sklearn NB':>12} {'Difference':>12}")
print(f"  {'-'*48}")
print(f"  {'Accuracy':<12} {acc_custom:>12.6f} {acc_sklearn:>12.6f} {abs(acc_custom - acc_sklearn):>12.6f}")
print(f"  {'Precision':<12} {prec_custom:>12.6f} {prec_sklearn:>12.6f} {abs(prec_custom - prec_sklearn):>12.6f}")
print(f"  {'Recall':<12} {rec_custom:>12.6f} {rec_sklearn:>12.6f} {abs(rec_custom - rec_sklearn):>12.6f}")
print(f"  {'F1 Score':<12} {f1_custom:>12.6f} {f1_sklearn:>12.6f} {abs(f1_custom - f1_sklearn):>12.6f}")

# Agreement between the two classifiers
agreement = np.mean(y_pred_custom == y_pred_sklearn)
print(f"\n  Prediction agreement: {agreement:.4%}")



Dataset size after dropping mmissing values: 5644
Class distribution: {'e': 3488, 'p': 2156}
Part 1: Prior probabilities (Custom NB)

  Class = Edible:
    P(Odor=a | Edible) = 0.113992
    P(Odor=c | Edible) = 0.000381
    P(Odor=f | Edible) = 0.000381
    P(Odor=l | Edible) = 0.112848
    P(Odor=m | Edible) = 0.000381
    P(Odor=n | Edible) = 0.771636
    P(Odor=p | Edible) = 0.000381

  Class = Poisonous:
    P(Odor=a | Poisonous) = 0.000616
    P(Odor=c | Poisonous) = 0.088054
    P(Odor=f | Poisonous) = 0.732143
    P(Odor=l | Poisonous) = 0.000616
    P(Odor=m | Poisonous) = 0.015394
    P(Odor=n | Poisonous) = 0.043103
    P(Odor=p | Poisonous) = 0.120074

Part 2: Predictions on test set (Custom NB)
    Total test samples: 1411
    Predicted Edible: 910
    Predicted Poisonous: 501

Part 3: Evaluation Metrics (Custom NB)
    Accuracy:  0.970234
    Precision: 0.996008
    Recall:    0.925788
    F1 Score:  0.959615

PART 4: Comparison with sklearn CategoricalNB

  Metric        